# TFM — Profesores grandes y destilación hacia mmBERT-small

Las celdas 0-5 son las del cuaderno principal (`tfm_comparativa.ipynb`). Desde la sección 6: profesores grandes
(XLM-R large y un LLM de 7 B) y destilación de su conocimiento en mmBERT-small, el modelo pequeño del escenario de aplicación.

**Antes de ejecutar:** Entorno → Cambiar tipo → **A100** si está disponible (con L4 también funciona, más lento).
Duración aproximada en A100: XLM-R large ~2 h, alumno ~1 h, LLM ~2 h, segundo alumno ~1 h. Si la sesión se corta,
volver a ejecutar: lo terminado se salta (pero las probabilidades de los profesores están en el disco local de Colab:
si se pierde la sesión hay que repetir esos entrenamientos).

# TFM — Comparativa de arquitecturas ligeras para anonimización contextual en el borde

Este cuaderno entrena y evalúa, con un protocolo idéntico, cuatro codificadores Transformer multilingües para
reconocer datos personales (NER de privacidad) en texto en español, inglés y mixto:

| Clave | Modelo | Papel en la comparativa |
|---|---|---|
| `distilbert-mbert` | `distilbert/distilbert-base-multilingual-cased` | DistilBERT multilingüe (sustituye a DistilBERT/TinyBERT/MobileBERT de la propuesta, que solo existen en inglés) |
| `minilm-l12` | `microsoft/Multilingual-MiniLM-L12-H384` | Destilación profunda, el más pequeño |
| `mmbert-small` | `jhu-clsp/mmBERT-small` | Arquitectura moderna (ModernBERT multilingüe, 2025) |
| `xlmr-base` | `FacebookAI/xlm-roberta-base` | Referencia de techo (no es ligero) |

**Idea general.** Cada modelo lee un mensaje y decide, token a token, si es parte de un dato personal y de qué tipo
(clasificación de tokens con etiquetas BIO). Todos parten de un modelo ya preentrenado en muchos idiomas y se
ajustan (*fine-tuning*, aprendizaje por transferencia) con nuestro corpus.

**Protocolo** (sección 1): 4 épocas, lote 32, AdamW con decaimiento lineal y 6 % de calentamiento, *weight decay* 0,01;
se guarda la mejor época según validación. Búsqueda de *learning rate* en una rejilla de 3 valores por modelo con la
semilla 42, elegida **solo con validación** (nunca con los tests); después, dos semillas más con el learning rate
elegido. Resultado final: media ± desviación típica sobre 3 semillas.

**Antes de ejecutar**
1. Entorno de ejecución → Cambiar tipo → **GPU A100**.
2. En Google Drive, carpeta `MyDrive/tfm_pii/`: `corpus_pii_es_en_v3.zip`.
3. Ejecutar de arriba abajo. Todo se guarda en `MyDrive/tfm_pii/tfm/` conforme termina; si se corta la sesión,
   volver a ejecutar: lo ya hecho se salta.

**Etiquetas**: `PER` persona, `LOC` lugar, `ID` identificador (DNI, pasaporte…), `FIN` dato financiero, `DEMO` dato
demográfico o de categoría especial (salud, religión…), `CON` contraseña o secreto, `CONTACT` email o teléfono, y
`PUB` personaje público mencionado como tema (se etiqueta pero **no** es dato personal: nunca cuenta como ocultado
ni como fuga). En formato BIO: `B-X` = primer token de un valor de tipo X, `I-X` = tokens siguientes, `O` = fuera.

**Métricas** (sobre tramos de caracteres; idénticas para PyTorch y ONNX):
- *strict P / R / F1* — el tramo y la etiqueta tienen que ser exactos (equivale a seqeval en modo estricto, a nivel
  de entidad); también por etiqueta, y *macro F1* = media de las 7 etiquetas de datos personales.
- *overlap R* — el valor se encuentra al menos en parte con la etiqueta correcta.
- **tasa de fuga** — proporción de caracteres de datos personales que quedan sin ocultar (con cualquier etiqueta).
  Es la métrica que más importa para la privacidad.
- **falsas alarmas** — proporción de mensajes *sin* datos personales en los que el modelo oculta algo.

## 0. Instalación

Comprueba qué GPU ha asignado Colab e instala lo que falta: `onnx` y `onnxruntime` (exportación e inferencia en
CPU) y una versión de `transformers` con mmBERT. Colab ya trae `datasets`, `accelerate` y `pandas`; no se
actualizan para no romper los paquetes propios de Colab (que exigen su versión de pandas).

In [ ]:
!nvidia-smi -L
!pip -q install "transformers>=4.48" onnx onnxruntime

## 1. Configuración

Todos los parámetros del experimento están aquí, en un solo sitio. Para repetir el experimento con otros valores
basta con cambiar esta celda.

In [ ]:
import os, json, re, random, time, math, collections, zipfile, shutil, gc, gzip, pickle, subprocess, sys
from pathlib import Path
import numpy as np

# --- Rutas
DRIVE_DIR = "/content/drive/MyDrive/tfm_pii"
DATA_ZIP  = f"{DRIVE_DIR}/corpus_pii_es_en_v3.zip"   # corpus (train, val y los 4 conjuntos de test)
RUNS_DIR  = f"{DRIVE_DIR}/tfm"            # resultados, curvas y probabilidades (persisten en Drive)
WORK_DIR  = "/content/work"               # disco local de Colab: checkpoints temporales (se pierden al cerrar)
# Mejor modelo de cada candidato (para exportarlo a ONNX). En el disco local de Colab, NO en Drive: los 4 modelos
# ocupan ~2,7 GB y la cuenta gratuita de Drive tiene 15 GB. Si la sesión se corta y se pierden, la sección 6 los
# vuelve a entrenar (mismo learning rate y semilla 42 = mismo modelo).
BEST_DIR  = "/content/best"

# --- Etiquetas. El modelo predice 17 clases: O + (B-, I-) x 8 tipos.
LABELS = ["PER", "LOC", "ID", "FIN", "DEMO", "CON", "CONTACT", "PUB"]
NON_PII = {"PUB"}                         # se etiqueta, pero no se oculta ni cuenta como fuga
ID2LABEL = {0: "O"}                       # número de clase -> nombre ("O", "B-PER", "I-PER", ...)
for i, l in enumerate(LABELS):
    ID2LABEL[2 * i + 1] = f"B-{l}"
    ID2LABEL[2 * i + 2] = f"I-{l}"
LABEL2ID = {v: k for k, v in ID2LABEL.items()}   # y al revés

# --- Candidatos. `model` = pesos preentrenados en Hugging Face; `tokenizer` = cómo se trocea el texto
# (MiniLM usa el de XLM-R: mismo vocabulario sobre un cuerpo BERT). `lr_grid` = 3 learning rates a probar,
# centrados en el valor recomendado por sus autores o en el usado en la prueba previa.
CANDIDATES = {
    "distilbert-mbert": dict(model="distilbert/distilbert-base-multilingual-cased",
                             tokenizer="distilbert/distilbert-base-multilingual-cased", lr_grid=[3e-5, 5e-5, 8e-5]),
    "minilm-l12":       dict(model="microsoft/Multilingual-MiniLM-L12-H384",
                             tokenizer="FacebookAI/xlm-roberta-base", lr_grid=[4e-5, 6e-5, 1e-4]),
    "mmbert-small":     dict(model="jhu-clsp/mmBERT-small", tokenizer="jhu-clsp/mmBERT-small",
                             lr_grid=[3e-5, 5e-5, 8e-5]),
    "xlmr-base":        dict(model="FacebookAI/xlm-roberta-base", tokenizer="FacebookAI/xlm-roberta-base",
                             lr_grid=[2e-5, 3e-5, 5e-5]),
}
RUN_CANDIDATES = list(CANDIDATES)         # se puede reducir, p. ej. ["mmbert-small"], para pruebas rápidas

# --- Protocolo de entrenamiento: IGUAL para todos los candidatos (es lo que hace justa la comparación)
EPOCHS, BATCH, WARMUP, WEIGHT_DECAY = 4, 32, 0.06, 0.01
SEEDS = [42, 13, 2026]                    # la primera se usa en la búsqueda de learning rate
DO_LR_SEARCH = True                       # False: usa el valor central de la rejilla para todas las semillas
MAX_LEN, STRIDE = 512, 128                # ventana máxima del modelo (tokens) y solape entre ventanas

# --- Conjuntos que se evalúan, y en cuáles se guardan las probabilidades por token para el análisis local
EVAL_SETS = ("val", "test", "test_long", "test_v2", "test_v3")
PROB_SETS = ("test", "test_long", "test_v2", "test_v3")
os.makedirs(WORK_DIR, exist_ok=True)

## 2. Datos

Monta Google Drive, crea las carpetas de resultados y descomprime el corpus. Conjuntos:
- `train` (75.098 mensajes) y `val` (4.345): entrenamiento y validación (elegir época y learning rate).
- `test_v1` (1.000), `test_long_v1` (150 textos de 1.500–3.400 caracteres), `test_v2` (395, escritos a mano y
  **nunca usados para ajustar nada**: es el test principal) y `test_v3` (102, personajes públicos, erratas,
  salud y dinero).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for sub in ("", "/models", "/logs", "/probs", "/onnx"):
    os.makedirs(RUNS_DIR + sub, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as z:
    z.extractall("/content")
DATA_DIR = "/content/corpus_pii_es_en_v3"

Funciones para leer el corpus. Cada línea de los ficheros `.jsonl` es un mensaje:
`{"text": "...", "entities": [{"start": 10, "end": 21, "label": "PER", "text": "Lucía Martín"}, ...]}`,
con las posiciones de cada dato personal en caracteres. También se definen dos "cortes" para analizar resultados
por subgrupos: el idioma del mensaje y si contiene valores "rotos" (escritos de forma que una regla fija no los
reconocería).

In [ ]:
def load_jsonl(path):
    # Lee un .jsonl y comprueba que las posiciones de cada entidad coinciden con su texto (detecta datos corruptos).
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    for r in rows:
        for e in r["entities"]:
            assert r["text"][e["start"]:e["end"]] == e.get("text", r["text"][e["start"]:e["end"]]), (r.get("id"), e)
    return rows


def row_lang(r):
    # Idioma del mensaje: "es", "en" o "mixed" (según el campo lang o los metadatos de generación).
    if r.get("lang"):
        return {"mix": "mixed"}.get(r["lang"], r["lang"])
    axes = (r.get("meta") or {}).get("axes") or {}
    return {"mix": "mixed"}.get(axes.get("lang"), axes.get("lang", "?"))


def is_broken_value(e):
    # ¿Valor que una regla fija no detectaría? Letras en vez de dígitos (12O45), "arroba" en vez de @,
    # separadores raros, asteriscos… Sirve para medir la robustez frente a texto "sucio".
    t, lab = e["text"], e["label"]
    if lab == "CONTACT":
        if re.search(r"arroba|\bat\b|\(at\)|\[at\]|\bAT\b|\bpunto\b|\bdot\b", t, re.I):
            return True
        if "@" in t:
            return bool(re.search(r"\s|@@|,|@[\w-]+$|@$|\.con$|gmai\.|gmial|hotmial|yahooo", t)) or not re.search(r"@[\w-]+\.\w", t)
        digits = re.sub(r"\D", "", t)
        return bool(re.search(r"[A-Za-z]", t)) or len(digits) not in (9, 11, 12, 13) or bool(re.search(r"\d \d \d", t))
    if lab in ("ID", "FIN", "LOC"):
        return bool(re.search(r"\d[OoIlS?]|[OoIlS?]\d", t)) or "*" in t or "\n" in t or bool(re.search(r"\d \d \d \d", t))
    return False


def row_broken(r):
    return any(is_broken_value(e) for e in r["entities"])


def dataset_stats(name, rows):
    # Una fila de la tabla de estadísticas del corpus (capítulo 4 de la memoria).
    labs = collections.Counter(e["label"] for r in rows for e in r["entities"])
    return {"set": name, "rows": len(rows), "entities": sum(labs.values()),
            "no_pii_share": round(sum(not any(e["label"] not in NON_PII for e in r["entities"]) for r in rows) / max(len(rows), 1), 3),
            "chars_mean": round(float(np.mean([len(r["text"]) for r in rows])), 1) if rows else 0,
            **{f"n_{l}": labs.get(l, 0) for l in LABELS},
            **{f"lang_{k}": v for k, v in collections.Counter(row_lang(r) for r in rows).items()}}

Carga los seis conjuntos y muestra (y guarda en Drive) sus estadísticas: mensajes, entidades por tipo, idiomas y proporción de mensajes sin datos personales.

In [ ]:
train_rows = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_rows   = load_jsonl(f"{DATA_DIR}/val.jsonl")
test_rows  = load_jsonl(f"{DATA_DIR}/test_v1.jsonl")
long_rows  = load_jsonl(f"{DATA_DIR}/test_long_v1.jsonl")
test2_rows = load_jsonl(f"{DATA_DIR}/test_v2.jsonl")
test3_rows = load_jsonl(f"{DATA_DIR}/test_v3.jsonl")
import pandas as pd
stats = pd.DataFrame([dataset_stats(n, r) for n, r in [("train", train_rows), ("val", val_rows), ("test_v1", test_rows),
                      ("test_long", long_rows), ("test_v2", test2_rows), ("test_v3", test3_rows)]]).fillna(0)
stats.to_csv(f"{RUNS_DIR}/dataset_stats.csv", index=False)
display(stats)

## 3. Tokenización y alineación de etiquetas

Los modelos no ven caracteres sino **tokens** (trozos de palabra: "Martín" puede ser "Mart" + "ín"). Hay que
traducir las etiquetas del corpus (en caracteres) a una etiqueta por token:
- un token recibe `B-X` si es el primero que solapa un valor de tipo X, `I-X` los siguientes, y `O` el resto;
- los tokens especiales (inicio/fin de secuencia) reciben -100, que el entrenamiento ignora.

Los modelos solo aceptan 512 tokens a la vez, así que los textos largos se dividen en **ventanas solapadas**
(512 tokens, solape de 128): no se trunca nada, y en la parte solapada se promedian las dos predicciones.

In [ ]:
_SPECIALS = {}


def _special_ids(tok):
    # Averigua qué tokens especiales pone el tokenizador antes y después de una secuencia (p. ej. <s> y </s>),
    # tokenizando "a" con y sin ellos. Se guarda en caché por tokenizador.
    key = id(tok)
    if key not in _SPECIALS:
        with_sp = tok("a")["input_ids"]
        body = tok("a", add_special_tokens=False)["input_ids"]
        i = next(i for i in range(len(with_sp)) if with_sp[i:i + len(body)] == body)
        _SPECIALS[key] = (with_sp[:i], with_sp[i + len(body):])
    return _SPECIALS[key]


def windows_for_text(tok, text, max_len=MAX_LEN, stride=STRIDE):
    # Trocea un texto en ventanas de como mucho `max_len` tokens que se solapan `stride` tokens.
    # Devuelve [(ids, máscara de atención, offsets)]; offsets = posición (inicio, fin) en caracteres de cada token,
    # o None para los tokens especiales. Se hace a mano porque transformers 5 pierde texto con su opción automática.
    pre, suf = _special_ids(tok)
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True, verbose=False)
    ids, offs = enc["input_ids"], [tuple(o) for o in enc["offset_mapping"]]
    body = max_len - len(pre) - len(suf)          # tokens de texto que caben en una ventana
    out, start = [], 0
    while True:
        w_ids = pre + ids[start:start + body] + suf
        w_offs = [None] * len(pre) + offs[start:start + body] + [None] * len(suf)
        out.append((w_ids, [1] * len(w_ids), w_offs))
        if start + body >= len(ids):
            return out
        start += body - stride                    # la siguiente ventana empieza `stride` tokens antes del final


def align_labels(offsets, entities):
    # Convierte las entidades (en caracteres) en una etiqueta numérica por token (B-/I-/O, o -100 si se ignora).
    labels, prev = [], None
    for o in offsets:
        if o is None or o[0] == o[1]:             # token especial o vacío: el entrenamiento lo ignora
            labels.append(-100)
            continue
        s, e = o
        hit = None
        for k, ent in enumerate(entities):        # ¿qué entidad solapa este token?
            if s < ent["end"] and e > ent["start"]:
                hit = k
                break
        if hit is None:
            labels.append(0)                      # O
            prev = None
        else:
            ent = entities[hit]
            # B- si es el primer token de esa entidad, I- si continúa la misma entidad que el token anterior
            labels.append(LABEL2ID[("I-" if prev == hit else "B-") + ent["label"]])
            prev = hit
    return labels


def encode_rows(rows, tok):
    # Prepara un conjunto de mensajes para el entrenador: ids, máscara y etiquetas por ventana.
    feats = {"input_ids": [], "attention_mask": [], "labels": []}
    for r in rows:
        ents = sorted(r["entities"], key=lambda e: e["start"])
        for ids, mask, offs in windows_for_text(tok, r["text"]):
            feats["input_ids"].append(ids)
            feats["attention_mask"].append(mask)
            feats["labels"].append(align_labels(offs, ents))
    return feats

## 4. Inferencia y métricas (comunes a PyTorch y ONNX)

Para poder comparar todo con la misma vara, la evaluación es igual para el modelo en GPU (PyTorch) y para el
modelo exportado en CPU (ONNX): ambos son un "predictor" con un método `logits(ids, máscara)`. El proceso:
1. `predict_probs`: ventanas → modelo → probabilidades por token (softmax), promediando las ventanas solapadas.
2. `spans_from_probs` / `decode_spans`: en cada token se elige la clase más probable y los tokens seguidos del
   mismo tipo se unen en un tramo de caracteres (`{"start", "end", "label"}`), al que se recortan espacios y signos.
3. `evaluate`: compara los tramos predichos con los anotados y calcula todas las métricas.

In [ ]:
class TorchPredictor:
    # Modelo de PyTorch en GPU (precisión bfloat16), para evaluar justo después de entrenar.
    def __init__(self, model_dir):
        import torch
        from transformers import AutoModelForTokenClassification, AutoTokenizer
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForTokenClassification.from_pretrained(model_dir).eval().cuda()
        self.id2label = {int(k): v for k, v in self.model.config.id2label.items()}

    def logits(self, ids, mask):
        with self.torch.no_grad(), self.torch.autocast("cuda", dtype=self.torch.bfloat16):
            out = self.model(input_ids=self.torch.tensor(ids).cuda(), attention_mask=self.torch.tensor(mask).cuda())
        return out.logits.float().cpu().numpy()


class OnnxPredictor:
    # Modelo exportado a ONNX, ejecutado en CPU con ONNX Runtime (como en el navegador o en un portátil).
    def __init__(self, onnx_path, tok_dir, threads=4):
        import onnxruntime as ort
        from transformers import AutoTokenizer
        so = ort.SessionOptions()
        so.intra_op_num_threads = threads         # hilos de CPU (la extensión del navegador usa 1)
        self.sess = ort.InferenceSession(onnx_path, so, providers=["CPUExecutionProvider"])
        self.tok = AutoTokenizer.from_pretrained(tok_dir)
        cfg = json.load(open(Path(tok_dir) / "config.json"))
        self.id2label = {int(k): v for k, v in cfg["id2label"].items()}
        self.inputs = {i.name for i in self.sess.get_inputs()}

    def logits(self, ids, mask):
        feed = {"input_ids": np.array(ids, dtype=np.int64), "attention_mask": np.array(mask, dtype=np.int64)}
        return self.sess.run(["logits"], {k: v for k, v in feed.items() if k in self.inputs})[0]


TRIM_LEAD, TRIM_TAIL = "([{¿¡\"'«", ".,;:)]}\"'»"


def trim_span(text, s, e, label):
    # Quita espacios y signos de puntuación de los bordes de un tramo ("(Lucía)," -> "Lucía").
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    if label != "CON":  # una contraseña sí puede empezar o acabar con signos de puntuación
        while e > s and text[e - 1] in TRIM_TAIL and not (text[e - 1] == ")" and "(" in text[s:e]):
            e -= 1
        while s < e and text[s] in TRIM_LEAD and not (text[s] == "(" and ")" in text[s:e]):
            s += 1
    return s, e


def decode_spans(text, tokens, id2label):
    # tokens: [(inicio, fin, clase)] ordenados por posición. Une tokens consecutivos B-X, I-X, I-X… en un tramo.
    spans, cur = [], None
    for s, e, lid in tokens:
        lab = id2label[lid]
        if lab == "O":                            # fuera de cualquier dato: cierra el tramo en curso
            if cur:
                spans.append(cur)
                cur = None
            continue
        bio, typ = lab.split("-", 1)
        if cur and typ == cur[2] and bio == "I":  # continúa el tramo del mismo tipo
            cur = (cur[0], e, typ)
        else:                                     # empieza un tramo nuevo
            if cur:
                spans.append(cur)
            cur = (s, e, typ)
    if cur:
        spans.append(cur)
    out = []
    for s, e, typ in spans:
        s, e = trim_span(text, s, e, typ)
        if e > s:
            out.append({"start": s, "end": e, "label": typ})
    return out


def predict_probs(rows, predictor, batch=64):
    # Para cada mensaje devuelve (offsets [n,2], probabilidades [n,17]) por token. Las ventanas de todos los mensajes
    # se ordenan por longitud y se agrupan en lotes (menos relleno = más rápido); al final se reparten por mensaje,
    # promediando los tokens que aparecen en dos ventanas solapadas.
    tok = predictor.tok
    windows = []
    for i, r in enumerate(rows):
        for ids, mask, offs in windows_for_text(tok, r["text"]):
            windows.append((i, ids, mask, offs))
    windows.sort(key=lambda w: len(w[1]))
    acc = collections.defaultdict(dict)           # mensaje -> {offset del token: [suma de probabilidades, nº de ventanas]}
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    for b in range(0, len(windows), batch):
        chunk = windows[b:b + batch]
        L = max(len(w[1]) for w in chunk)
        ids = [w[1] + [pad] * (L - len(w[1])) for w in chunk]          # relleno hasta la ventana más larga del lote
        mask = [w[2] + [0] * (L - len(w[2])) for w in chunk]
        lg = predictor.logits(ids, mask).astype(np.float32)
        p = np.exp(lg - lg.max(-1, keepdims=True))                     # softmax: logits -> probabilidades
        p /= p.sum(-1, keepdims=True)
        for k, (ri, _, _, offs) in enumerate(chunk):
            for t, o in enumerate(offs):
                if o is None or o[0] == o[1]:
                    continue
                a = acc[ri].setdefault(o, [0.0, 0])
                a[0] = a[0] + p[k, t]
                a[1] += 1
    out = []
    n_cls = len(predictor.id2label)
    for i in range(len(rows)):
        items = sorted(acc[i].items())
        offs = np.array([o for o, _ in items], dtype=np.int32).reshape(-1, 2)
        probs = np.array([v[0] / v[1] for _, v in items], dtype=np.float32).reshape(-1, n_cls)
        out.append((offs, probs))
    return out


def spans_from_probs(rows, probs, id2label):
    # Clase más probable por token (argmax) -> tramos de caracteres.
    preds = []
    for r, (offs, p) in zip(rows, probs):
        toks = [(int(s), int(e), int(k)) for (s, e), k in zip(offs, p.argmax(-1))] if len(offs) else []
        preds.append(decode_spans(r["text"], toks, id2label))
    return preds


def predict_rows(rows, predictor, batch=64):
    return spans_from_probs(rows, predict_probs(rows, predictor, batch), predictor.id2label)


def evaluate(rows, preds):
    # Compara predicciones y anotaciones de un conjunto de mensajes. Devuelve un diccionario de métricas.
    tp = collections.Counter(); fp = collections.Counter(); fn = collections.Counter()   # por etiqueta
    ov_hit = collections.Counter(); ov_tot = collections.Counter()
    gold_chars = leaked = 0
    leak_by = collections.Counter(); chars_by = collections.Counter()
    neg_rows = neg_alarm = 0
    per_chars = per_as_pub = pub_chars = pub_masked = 0
    for r, P in zip(rows, preds):
        G = r["entities"]
        # 1) Coincidencia estricta: mismo inicio, fin y etiqueta.
        gset = {(e["start"], e["end"], e["label"]) for e in G}
        pset = {(e["start"], e["end"], e["label"]) for e in P}
        for x in pset & gset: tp[x[2]] += 1       # acierto
        for x in pset - gset: fp[x[2]] += 1       # predicho pero no anotado
        for x in gset - pset: fn[x[2]] += 1       # anotado pero no predicho
        # 2) Fuga: qué caracteres quedan ocultos (cualquier etiqueta salvo PUB, que nunca se oculta).
        covered = np.zeros(len(r["text"]) + 1, dtype=bool)
        as_pub = np.zeros(len(r["text"]) + 1, dtype=bool)
        for e in P:
            (as_pub if e["label"] in NON_PII else covered)[e["start"]:e["end"]] = True
        for e in G:
            ov_tot[e["label"]] += 1
            if any(p["label"] == e["label"] and p["start"] < e["end"] and p["end"] > e["start"] for p in P):
                ov_hit[e["label"]] += 1           # encontrado al menos en parte con la etiqueta correcta
            n = e["end"] - e["start"]
            if e["label"] in NON_PII:             # personaje público: ¿se ha ocultado de más?
                pub_chars += n; pub_masked += int(covered[e["start"]:e["end"]].sum())
                continue
            miss = int(n - covered[e["start"]:e["end"]].sum())         # caracteres de este dato que se escapan
            gold_chars += n; leaked += miss
            chars_by[e["label"]] += n; leak_by[e["label"]] += miss
            if e["label"] == "PER":               # persona confundida con personaje público (no se oculta)
                per_chars += n; per_as_pub += int((as_pub[e["start"]:e["end"]] & ~covered[e["start"]:e["end"]]).sum())
        # 3) Falsa alarma: mensaje sin datos personales en el que se oculta algo.
        if not any(e["label"] not in NON_PII for e in G):
            neg_rows += 1
            neg_alarm += any(p["label"] not in NON_PII for p in P)
    res = {}
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    res["strict_P"] = TP / max(TP + FP, 1)                         # precisión
    res["strict_R"] = TP / max(TP + FN, 1)                         # exhaustividad (recall)
    res["strict_F1"] = 2 * TP / max(2 * TP + FP + FN, 1)           # media armónica de las dos
    f1s = []
    for l in LABELS:
        d = 2 * tp[l] + fp[l] + fn[l]
        res[f"P_{l}"] = tp[l] / (tp[l] + fp[l]) if (tp[l] + fp[l]) else float("nan")
        res[f"R_{l}"] = tp[l] / (tp[l] + fn[l]) if (tp[l] + fn[l]) else float("nan")
        res[f"F1_{l}"] = 2 * tp[l] / d if d else float("nan")     # nan = la etiqueta no aparece en este conjunto
        if d and l not in NON_PII:
            f1s.append(res[f"F1_{l}"])
        if l not in NON_PII:
            res[f"leak_{l}"] = leak_by[l] / chars_by[l] if chars_by[l] else float("nan")
    res["macro_F1"] = float(np.mean(f1s)) if f1s else float("nan")   # media de las etiquetas de datos personales
    res["overlap_R"] = sum(ov_hit.values()) / max(sum(ov_tot.values()), 1)
    res["leak_rate"] = leaked / max(gold_chars, 1)
    res["false_alarm"] = neg_alarm / neg_rows if neg_rows else float("nan")
    res["per_as_pub"] = per_as_pub / per_chars if per_chars else float("nan")
    res["pub_masked"] = pub_masked / pub_chars if pub_chars else float("nan")
    res["n_rows"] = len(rows)
    return res


def rows_for(s):
    return {"val": val_rows, "test": test_rows, "test_long": long_rows, "test_v2": test2_rows, "test_v3": test3_rows}[s]


def save_probs(name, s, probs):
    # Guarda en Drive las probabilidades por token (en float16 para que ocupen la mitad). Con ellas se puede, en
    # local y sin GPU: cambiar el umbral de decisión, aplicar el post-proceso y hacer pruebas estadísticas.
    with gzip.open(f"{RUNS_DIR}/probs/{name}__{s}.pkl.gz", "wb") as f:
        pickle.dump([(o, p.astype(np.float16)) for o, p in probs], f)


def evaluate_all(predictor, name, extra=None, sets=EVAL_SETS, dump_probs=True):
    # Evalúa un modelo en varios conjuntos (y en sus cortes por idioma y por valores rotos) y añade cada resultado
    # como una línea en results.jsonl. `extra` = datos del experimento (modelo, learning rate, semilla, tamaño…).
    out = []
    for s in sets:
        rows = rows_for(s)
        if not rows:
            continue
        t0 = time.time()
        probs = predict_probs(rows, predictor)
        dt = time.time() - t0
        preds = spans_from_probs(rows, probs, predictor.id2label)
        if dump_probs and s in PROB_SETS:
            save_probs(name, s, probs)
        out.append({"run": name, "set": s, "secs": round(dt, 1), **(extra or {}), **evaluate(rows, preds)})
        if s in ("test", "test_v2", "test_v3"):   # cortes: test_v2[es], test_v2[en], test_v2[mixed], test_v2[broken]
            for lang in ("es", "en", "mixed"):
                idx = [i for i, r in enumerate(rows) if row_lang(r) == lang]
                if idx:
                    out.append({"run": name, "set": f"{s}[{lang}]", **(extra or {}),
                                **evaluate([rows[i] for i in idx], [preds[i] for i in idx])})
            idx = [i for i, r in enumerate(rows) if row_broken(r)]
            if idx:
                out.append({"run": name, "set": f"{s}[broken]", **(extra or {}),
                            **evaluate([rows[i] for i in idx], [preds[i] for i in idx])})
    with open(f"{RUNS_DIR}/results.jsonl", "a") as f:
        for rec in out:
            f.write(json.dumps(rec) + "\n")
    return out


def load_results():
    p = f"{RUNS_DIR}/results.jsonl"
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []


def run_done(name):
    # ¿Este experimento ya terminó en una sesión anterior? (permite reanudar tras una desconexión)
    return any(r["run"] == name and r["set"] == "val" for r in load_results())


def show(records, cols=("run", "set", "strict_F1", "macro_F1", "overlap_R", "leak_rate", "false_alarm",
                        "F1_PER", "F1_LOC", "F1_ID", "F1_FIN", "F1_DEMO", "F1_CON", "F1_CONTACT", "F1_PUB")):
    # Tabla compacta para ver resultados en el cuaderno.
    import pandas as pd
    df = pd.DataFrame(records)
    cols = [c for c in cols if c in df.columns] + [c for c in ("size_mb", "ram_mb", "ms_1t_1000ch") if c in df.columns]
    return df[cols].round(4)

## 5. Entrenamiento

`train_model` ajusta un candidato con el protocolo común usando el `Trainer` de Hugging Face:
- añade al modelo preentrenado una capa de clasificación por token (17 clases) que empieza aleatoria;
- en cada época evalúa en validación (*macro F1* por token) y al final conserva la mejor época;
- guarda en Drive el historial de pérdidas y métricas (para las **curvas de aprendizaje** de la memoria).

`train_and_eval` entrena y evalúa en todos los conjuntos, y se salta lo que ya está hecho.

In [ ]:
import inspect
import torch
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification, set_seed)
from datasets import Dataset
torch.backends.cuda.matmul.allow_tf32 = True      # multiplicaciones más rápidas en A100 sin pérdida apreciable


def compute_metrics(p):
    # Métrica rápida por token que usa el Trainer para elegir la mejor época (no es la métrica final del TFM).
    # Junta B-X e I-X en la misma clase X y calcula el F1 de cada tipo y la media.
    preds, labels = p.predictions.argmax(-1), p.label_ids
    m = labels != -100
    pt = np.where(preds > 0, (preds + 1) // 2, 0)[m]
    lt = np.where(labels > 0, (labels + 1) // 2, 0)[m]
    f1 = []
    for t in range(1, len(LABELS) + 1):
        tp = ((pt == t) & (lt == t)).sum(); fp = ((pt == t) & (lt != t)).sum(); fn = ((pt != t) & (lt == t)).sum()
        f1.append(2 * tp / max(2 * tp + fp + fn, 1))
    pii_recall = ((pt > 0) & (lt > 0)).sum() / max((lt > 0).sum(), 1)
    return {"tok_macro_f1": float(np.mean(f1)), "tok_pii_recall": float(pii_recall)}


_TA_PARAMS = inspect.signature(TrainingArguments.__init__).parameters
# lotes de longitud parecida = menos relleno (transformers 5 renombró group_by_length)
GROUP_BY_LENGTH = ({"train_sampling_strategy": "group_by_length"} if "train_sampling_strategy" in _TA_PARAMS
                   else {"group_by_length": True})


def count_params(model):
    # Parámetros totales y cuántos son de la tabla de embeddings (el vocabulario): explica buena parte del tamaño.
    total = sum(p.numel() for p in model.parameters())
    emb = model.get_input_embeddings().weight.numel()
    return total, emb


def train_model(key, lr, seed, run_name, trainer_cls=None, ds_hook=None):
    # Devuelve (carpeta del modelo final, información del entrenamiento). Mismo protocolo para todos los candidatos.
    # Opciones de CANDIDATES[key] para modelos grandes (si faltan, el protocolo común de siempre):
    #   per_device  lote por GPU; el resto se acumula hasta BATCH (mismo lote efectivo, menos memoria)
    #   epochs      épocas (el LLM de 7 B usa menos, por coste)
    #   model_kwargs  argumentos de carga (p. ej. dtype bf16 para el LLM)
    #   prepare     función (modelo, tokenizador) -> modelo: añade LoRA, atención bidireccional…
    #   grad_ckpt   gradient checkpointing (recalcula activaciones para ahorrar memoria)
    # trainer_cls y ds_hook los usa la destilación: otro Trainer (pérdida con el profesor) y otro conjunto de
    # entrenamiento (con el índice de cada ventana para buscar sus probabilidades del profesor).
    cfg = CANDIDATES[key]
    set_seed(seed)                                 # misma semilla = mismo resultado (reproducibilidad)
    tok = AutoTokenizer.from_pretrained(cfg["tokenizer"])
    t0 = time.time()
    ds_tr = Dataset.from_dict(ds_hook(train_rows, tok) if ds_hook else encode_rows(train_rows, tok))
    ds_va = Dataset.from_dict(encode_rows(val_rows, tok))
    print(f"[{run_name}] ventanas: train {len(ds_tr)}, val {len(ds_va)} (tokenizado en {time.time()-t0:.0f}s)")
    # modelo preentrenado + capa de clasificación nueva con nuestras 17 clases
    model = AutoModelForTokenClassification.from_pretrained(
        cfg["model"], num_labels=len(ID2LABEL), id2label=ID2LABEL, label2id=LABEL2ID, **cfg.get("model_kwargs", {}))
    n_params, n_emb = count_params(model)
    vocab = int(model.get_input_embeddings().weight.shape[0])
    if "prepare" in cfg:
        model = cfg["prepare"](model, tok)
    out_dir = f"{WORK_DIR}/{run_name}"
    per_device = cfg.get("per_device", BATCH)
    epochs = cfg.get("epochs", EPOCHS)
    warmup = int(WARMUP * math.ceil(len(ds_tr) / BATCH) * epochs)   # 6 % de los pasos con learning rate creciente
    if cfg.get("ckpt_dir"):
        # Entrenamientos largos (el LLM): puntos de control cada `save_steps` pasos EN DRIVE, para continuar donde iba
        # si Colab corta el entorno. Sin evaluación por época (es una sola época: no hay época que elegir).
        ckpt_dir = f"{cfg['ckpt_dir']}/{run_name}"
        schedule = dict(output_dir=ckpt_dir, eval_strategy="no", save_strategy="steps",
                        save_steps=cfg.get("save_steps", 200), save_total_limit=1)
    else:
        ckpt_dir = None
        schedule = dict(output_dir=out_dir, eval_strategy="epoch", save_strategy="epoch",
                        load_best_model_at_end=True,     # evaluar cada época y quedarse con la mejor
                        metric_for_best_model="tok_macro_f1", greater_is_better=True, save_total_limit=1)
    args = TrainingArguments(
        per_device_train_batch_size=per_device, gradient_accumulation_steps=BATCH // per_device,
        per_device_eval_batch_size=cfg.get("eval_batch", 64),
        learning_rate=lr, num_train_epochs=epochs, weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup, lr_scheduler_type="linear", bf16=True,          # bf16: más rápido en A100
        gradient_checkpointing=cfg.get("grad_ckpt", False), remove_unused_columns=ds_hook is None,
        logging_steps=50, report_to="none", seed=seed, dataloader_num_workers=2, **schedule, **GROUP_BY_LENGTH,
        **({"data_seed": seed} if "data_seed" in _TA_PARAMS else {}))
    trainer = (trainer_cls or Trainer)(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                                       data_collator=DataCollatorForTokenClassification(tok), processing_class=tok,
                                       compute_metrics=compute_metrics)
    resume = None
    if ckpt_dir and os.path.isdir(ckpt_dir):
        found = sorted(Path(ckpt_dir).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
        resume = str(found[-1]) if found else None
        if resume:
            print(f"[{run_name}] se continúa desde {resume}")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=resume)
    minutes = (time.time() - t0) / 60
    print(f"[{run_name}] entrenado en {minutes:.1f} min")
    json.dump(trainer.state.log_history, open(f"{RUNS_DIR}/logs/{run_name}.json", "w"))   # curvas de aprendizaje
    if ckpt_dir:                                   # terminado: el punto de control de Drive ya no hace falta
        shutil.rmtree(ckpt_dir, ignore_errors=True)
    best_epoch = None                              # qué época resultó la mejor en validación
    if trainer.state.best_model_checkpoint:
        step = int(trainer.state.best_model_checkpoint.rstrip("/").split("-")[-1])
        best_epoch = next((h["epoch"] for h in trainer.state.log_history if h.get("step") == step and "eval_loss" in h), None)
    final = f"{out_dir}/final"
    if hasattr(trainer.model, "merge_and_unload"):  # LoRA: se suman los adaptadores al modelo y se guarda completo
        trainer.model.merge_and_unload().save_pretrained(final)
    else:
        trainer.save_model(final)
    tok.save_pretrained(final)
    info = {"model": key, "lr": lr, "seed": seed, "train_min": round(minutes, 2), "best_epoch": best_epoch,
            "params_m": round(n_params / 1e6, 1), "emb_params_m": round(n_emb / 1e6, 1),
            "vocab": vocab, "train_windows": len(ds_tr)}
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()         # liberar memoria de la GPU antes del siguiente modelo
    return final, info


def train_and_eval(key, lr, seed):
    # Entrena un candidato con un learning rate y una semilla, y lo evalúa en todos los conjuntos.
    name = f"{key}-lr{lr:g}-s{seed}"               # p. ej. "mmbert-small-lr5e-05-s42"
    if run_done(name):
        print("ya hecho:", name)
        return name, None
    final, info = train_model(key, lr, seed, name)
    pred = TorchPredictor(final)
    evaluate_all(pred, name, {**info, "fmt": "torch-bf16"})
    del pred; gc.collect(); torch.cuda.empty_cache()
    return name, final

## 6. Destilación: configuración

**Qué es destilar.** Un modelo grande ya entrenado (el *profesor*) da, para cada token, una probabilidad por clase
("70 % salud, 25 % nada…"). Un modelo pequeño (el *alumno*) se entrena con dos objetivos a la vez:
- la pérdida normal con las etiquetas del corpus (entropía cruzada), y
- parecerse a las probabilidades del profesor (divergencia KL).

Esas probabilidades llevan más información que la etiqueta sola (qué casos son dudosos y con qué se confunden): es el
"conocimiento oscuro" de Hinton et al. (2015). La **temperatura** T suaviza ambas distribuciones para que el alumno vea
también las clases poco probables; la pérdida KL se multiplica por T² para que pese lo mismo con cualquier T.
Valores de la literatura: T = 2 y mitad y mitad entre las dos pérdidas (α = 0,5).

In [ ]:
!pip -q install peft
!pip -q uninstall -y torchao    # peft rechaza el torchao 0.10 de Colab; es opcional y aquí no se usa
TEACH_DIR = f"{RUNS_DIR}/teachers"          # probabilidades de los profesores, EN DRIVE (~150 MB por profesor):
                                           # si Colab corta el entorno no se pierden (el 5-nov se perdieron así)
os.makedirs(TEACH_DIR, exist_ok=True)
KD_T, KD_ALPHA = 2.0, 0.5                  # temperatura y peso de la pérdida normal (1 - α = peso de la destilación)
STUDENT_KEY, STUDENT_LR = "mmbert-small", 3e-5   # alumno: el modelo del escenario de aplicación, con el learning rate que eligió en validación
DUMP_TEACHER_PROBS_TO_DRIVE = False        # Drive está casi lleno: las probabilidades de los profesores solo en local

### 6.1 Profesores grandes

**XLM-R large** (Conneau et al., 2020): 24 capas, 550 M parámetros. Es el mejor modelo en el trabajo más parecido a
este (Graehl, 2026: etiquetado de privacidad en 35 idiomas). Los modelos *large* son sensibles al learning rate
(Mosbach et al., 2021), así que se prueban 1e-5 y 2e-5. Lote por GPU de 16 con acumulación de 2 = mismo lote efectivo (32).

**Qwen2.5-7B como etiquetador de tokens.** Un LLM normal solo "mira a la izquierda" (máscara causal): para etiquetar el
token *i* no ve lo que viene después, y eso perjudica al NER. Li et al. (2023) y Dukić y Šnajder (2024) quitan esa
máscara al ajustar el modelo y mejoran mucho. Aquí se hace igual: capa de clasificación por token + LoRA (Hu et al.,
2022: solo se entrenan matrices pequeñas añadidas, ~0,5 % de los parámetros) + atención bidireccional. 2 épocas en vez
de 4 por coste (en una L4, cada época de un modelo de 7 B tarda ~2,7 h, lo que ~10 de mmBERT-small); con puntos de
control en Drive cada 200 pasos para poder continuar si Colab corta el entorno.

In [ ]:
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {GPU_GB:.0f} GB")


def bidirectional(model):
    # Quita la máscara causal del LLM: cada token ve también los tokens de su derecha.
    # Cómo: (1) se marca cada capa de atención como no causal y (2) antes de entrar al modelo se convierte la máscara
    # de relleno [lote, longitud] en una máscara 4D [lote, 1, longitud, longitud] que solo tapa el relleno.
    # transformers usa una máscara 4D tal cual, sin añadirle la parte causal.
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    inner = base.model                                  # el cuerpo del LLM (sin la capa de clasificación)
    for m in inner.modules():
        if hasattr(m, "is_causal"):
            m.is_causal = False

    def hook(module, args, kwargs):
        am = kwargs.get("attention_mask")
        if am is not None and am.dim() == 2:
            b, L = am.shape
            kwargs["attention_mask"] = am[:, None, None, :].bool().expand(b, 1, L, L)
        return args, kwargs

    if not getattr(inner, "_bidir_hook", False):
        inner.register_forward_pre_hook(hook, with_kwargs=True)
        inner._bidir_hook = True
    return model


def is_bidirectional(model, tok):
    # Comprobación: si cambio la ÚLTIMA palabra, ¿cambia la salida del PRIMER token? Solo pasa si ve a su derecha.
    dev = next(model.parameters()).device
    a = tok("Hola Ana, tu DNI es 12345678Z", return_tensors="pt").to(dev)
    b = tok("Hola Ana, tu DNI es 87654321X", return_tensors="pt").to(dev)
    with torch.no_grad():
        la = model(input_ids=a["input_ids"], attention_mask=a["attention_mask"]).logits[0, 0].float()
        lb = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits[0, 0].float()
    return not torch.allclose(la, lb, atol=1e-4)


def prepare_llm(model, tok):
    # LLM -> etiquetador de tokens bidireccional con LoRA.
    from peft import LoraConfig, get_peft_model
    model.config.pad_token_id = tok.pad_token_id
    if torch.cuda.is_available():
        model = model.cuda()
    model = bidirectional(model)
    assert is_bidirectional(model, tok), "la atención sigue siendo causal: revisar la versión de transformers"
    print("atención bidireccional: OK")
    lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="TOKEN_CLS",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lcfg)               # la capa de clasificación nueva se entrena entera (task_type)
    model.enable_input_require_grads()                # necesario con gradient checkpointing
    model.print_trainable_parameters()
    return model


class LLMPredictor(TorchPredictor):
    # Igual que TorchPredictor, pero carga el LLM en bf16 y le vuelve a quitar la máscara causal.
    def __init__(self, model_dir):
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForTokenClassification.from_pretrained(model_dir, dtype=torch.bfloat16).eval().cuda()
        bidirectional(self.model)
        assert is_bidirectional(self.model, self.tok)
        self.id2label = {int(k): v for k, v in self.model.config.id2label.items()}


TEACHER_A = "xlmr-large"
CANDIDATES[TEACHER_A] = dict(model="FacebookAI/xlm-roberta-large", tokenizer="FacebookAI/xlm-roberta-large",
                             lr_grid=[1e-5, 2e-5], per_device=16)
TEACHER_B = "qwen2.5-7b"
CANDIDATES[TEACHER_B] = dict(model="Qwen/Qwen2.5-7B", tokenizer="Qwen/Qwen2.5-7B", lr_grid=[1e-4], epochs=2,
                             ckpt_dir=f"{RUNS_DIR}/ckpt", save_steps=200,   # puntos de control en Drive (~14 min)
                             per_device=8 if GPU_GB >= 40 else 4, eval_batch=16, grad_ckpt=True,
                             model_kwargs={"dtype": torch.bfloat16}, prepare=prepare_llm)

Funciones de los profesores: entrenar, evaluar (como un candidato más) y guardar sus probabilidades por token en
**train**, **val** y los tests (en el disco local). Las de train son las que aprende el alumno; las de val sirven para
elegir el conjunto de profesores sin mirar los tests.

In [ ]:
TEACH_SETS = ("train", "val", "test", "test_long", "test_v2", "test_v3")


def rows_any(s):
    return train_rows if s == "train" else rows_for(s)


def teacher_file(name, s):
    return f"{TEACH_DIR}/{name}__{s}.pkl.gz"


def teacher_ready(name):
    return run_done(name) and all(os.path.exists(teacher_file(name, s)) for s in TEACH_SETS)


def train_teacher(key, lr, seed, predictor_cls=None):
    name = f"{key}-lr{lr:g}-s{seed}"
    if teacher_ready(name):
        print("ya hecho:", name)
        return name
    final, info = train_model(key, lr, seed, name)
    pred = (predictor_cls or TorchPredictor)(final)
    evaluate_all(pred, name, {**info, "fmt": "torch-bf16", "role": "teacher"}, dump_probs=DUMP_TEACHER_PROBS_TO_DRIVE)
    for s in TEACH_SETS:                               # probabilidades del profesor, guardadas en local
        t0 = time.time()
        probs = predict_probs(rows_any(s), pred, batch=CANDIDATES[key].get("eval_batch", 64))
        with gzip.open(teacher_file(name, s), "wb") as f:
            pickle.dump([(o, p.astype(np.float16)) for o, p in probs], f)
        print(f"  probabilidades de {name} en {s}: {time.time() - t0:.0f} s")
    del pred; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(Path(final).parent, ignore_errors=True)   # el modelo ya no hace falta: sus probabilidades sí
    return name


def load_teacher(name, s):
    return pickle.load(gzip.open(teacher_file(name, s)))


def val_score(name):                                   # igual que en el cuaderno principal (sección 5a)
    r = [x for x in load_results() if x["run"] == name and x["set"] == "val"]
    return r[-1]["strict_F1"] if r else None


def best_lr_for(key):
    # El learning rate de la rejilla con mejor strict F1 en validación (semilla 42).
    scores = {lr: val_score(f"{key}-lr{lr:g}-s{SEEDS[0]}") for lr in CANDIDATES[key]["lr_grid"]}
    scores = {k: v for k, v in scores.items() if v is not None}
    return max(scores, key=scores.get) if scores else None

Probabilidades por **carácter**. Cada modelo trocea el texto a su manera (XLM-R, Qwen y mmBERT tienen tokenizadores
distintos), así que para combinar profesores y pasárselos al alumno se usa el carácter como unidad común:
- cada carácter recibe las 17 probabilidades del token del profesor que lo contiene;
- se apunta si ese carácter es el **inicio** de un token del profesor.

Para un token del alumno se toman las probabilidades de su primer carácter (sin contar espacios). Si ese carácter está
en mitad de un token del profesor (el alumno ha partido una palabra que el profesor tenía entera), la probabilidad de
"B-" (inicio de entidad) se pasa a "I-" (continuación): un trozo del medio de una palabra nunca empieza una entidad.
Con varios profesores se promedian sus probabilidades (conjunto, como en Hinton et al., 2015).

In [ ]:
B_IDX = [2 * i + 1 for i in range(len(LABELS))]       # columnas B-X
I_IDX = [2 * i + 2 for i in range(len(LABELS))]       # columnas I-X


def char_table(text, offs, probs):
    cp = np.zeros((len(text) + 1, len(ID2LABEL)), dtype=np.float32)
    cp[:, 0] = 1.0                                      # por defecto "O"
    start = np.zeros(len(text) + 1, dtype=bool)
    for (a, b), p in zip(offs, probs):
        cp[a:b] = p
        start[a] = True
    return cp, start


def soft_for_offsets(text, offs, tables):
    # Probabilidades objetivo (17 por token) para una ventana del alumno; los tokens especiales quedan a cero.
    out = np.zeros((len(offs), len(ID2LABEL)), dtype=np.float32)
    for t, o in enumerate(offs):
        if o is None or o[0] == o[1]:
            continue
        a, b = o
        c = next((k for k in range(a, b) if not text[k].isspace()), None)
        acc = np.zeros(len(ID2LABEL), dtype=np.float32)
        for cp, start in tables:
            if c is None:
                # Token que solo es espacio (mmBERT a veces separa el espacio de dentro de un valor: "ES74 0081").
                # Pertenece a la entidad si el profesor dice que lo que viene después CONTINÚA esa entidad (I-X);
                # si lo que sigue empieza otra (B-X) o no es nada (O), el espacio es O. Corrección del 5-nov: antes
                # recibía siempre O y el alumno aprendía a partir IBAN, teléfonos y direcciones en trozos.
                k = next((j for j in range(b, len(text)) if not text[j].isspace()), None)
                q = np.zeros(len(ID2LABEL), dtype=np.float32)
                if k is None:
                    q[0] = 1.0
                else:
                    nxt = cp[k]
                    q[I_IDX] = nxt[I_IDX] if start[k] else nxt[I_IDX] + nxt[B_IDX]
                    q[0] = 1.0 - q[I_IDX].sum()
            else:
                q = cp[c].copy()
                if not start[c]:                       # mitad de un token del profesor: B- -> I-
                    q[I_IDX] += q[B_IDX]
                    q[B_IDX] = 0
            acc += q
        out[t] = acc / len(tables)
    return out


def char_type_probs(text, offs, probs):
    # Para combinar profesores al EVALUAR: por carácter, P(O) y P(tipo) = P(B-tipo) + P(I-tipo), y P(inicio).
    c = np.zeros((len(text), len(LABELS) + 1), np.float32)
    c[:, 0] = 1
    b = np.zeros(len(text), np.float32)
    q = probs.astype(np.float32)
    t = np.concatenate([q[:, :1], q[:, B_IDX] + q[:, I_IDX]], 1)
    for (a, e), v, qq in zip(offs, t, q):
        c[a:e] = v
        ent = qq[1:].sum()
        b[a] = qq[B_IDX].sum() / ent if ent > 0 else 0
    return c, b


def char_spans(text, c, b):
    # Tramos a partir de las probabilidades por carácter: tipo más probable por carácter; dos tramos del mismo tipo
    # separados solo por espacios se unen si el segundo es continuación (P(inicio) < 0,5).
    lab = c.argmax(1)
    runs, i = [], 0
    while i < len(text):
        if lab[i] == 0:
            i += 1
            continue
        j = i
        while j < len(text) and lab[j] == lab[i]:
            j += 1
        runs.append([i, j, lab[i]])
        i = j
    merged = []
    for r in runs:
        if merged and merged[-1][2] == r[2] and text[merged[-1][1]:r[0]].isspace() and b[r[0]] < 0.5:
            merged[-1][1] = r[1]
        else:
            merged.append(r)
    out = []
    for s, e, l in merged:
        s, e = trim_span(text, s, e, LABELS[l - 1])
        if e > s:
            out.append({"start": s, "end": e, "label": LABELS[l - 1]})
    return out


def ensemble_eval(rows, prob_lists):
    # Evalúa el conjunto (media por carácter) de varios modelos. prob_lists = [probs del modelo 1, del 2, …].
    preds = []
    for k, r in enumerate(rows):
        cs = [char_type_probs(r["text"], *pl[k]) for pl in prob_lists]
        preds.append(char_spans(r["text"], np.mean([x[0] for x in cs], 0), np.mean([x[1] for x in cs], 0)))
    return evaluate(rows, preds)

El alumno: mismo `train_model` que el resto, con dos cambios.
- `kd_dataset` construye las ventanas de entrenamiento igual que siempre y guarda, para cada una, las probabilidades
  objetivo del conjunto de profesores (en la lista `SOFT`; el conjunto de datos solo lleva el índice `idx`).
- `KDTrainer` cambia la pérdida: α · entropía cruzada con las etiquetas + (1 − α) · T² · KL(profesor ‖ alumno).
  En la evaluación (validación) no hay `idx` y se usa la pérdida normal.

In [ ]:
from transformers import Trainer
SOFT = []                                              # probabilidades objetivo por ventana del alumno
_KD_CACHE = {}                                         # mismas ventanas para las 3 semillas: se calculan una vez


def kd_dataset(teacher_runs):
    def hook(rows, tok):
        key = (tuple(teacher_runs), tok.name_or_path, len(rows))
        if key in _KD_CACHE:
            feats, soft = _KD_CACHE[key]
            SOFT[:] = soft
            return feats
        T = [load_teacher(n, "train") for n in teacher_runs]
        SOFT.clear()
        feats = {"input_ids": [], "attention_mask": [], "labels": [], "idx": []}
        for ri, r in enumerate(rows):
            tables = [char_table(r["text"], *t[ri]) for t in T]
            ents = sorted(r["entities"], key=lambda e: e["start"])
            for ids, mask, offs in windows_for_text(tok, r["text"]):
                feats["input_ids"].append(ids)
                feats["attention_mask"].append(mask)
                feats["labels"].append(align_labels(offs, ents))
                feats["idx"].append(len(SOFT))
                SOFT.append(soft_for_offsets(r["text"], offs, tables).astype(np.float16))
        _KD_CACHE.clear()                              # solo se guarda el último conjunto (memoria)
        _KD_CACHE[key] = (feats, list(SOFT))
        return feats
    return hook


class KDTrainer(Trainer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        base = self.data_collator

        def collate(features):                         # saca idx antes de rellenar y lo vuelve a añadir como tensor
            idx = [f.pop("idx") for f in features] if "idx" in features[0] else None
            batch = base(features)
            if idx is not None:
                batch["idx"] = torch.tensor(idx)
            return batch
        self.data_collator = collate

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        idx = inputs.pop("idx", None)
        out = model(**inputs)
        if idx is None:                                # evaluación: pérdida normal
            return (out.loss, out) if return_outputs else out.loss
        logits = out.logits.float()
        soft = torch.zeros_like(logits)
        for b, i in enumerate(idx.tolist()):          # relleno a la derecha: las posiciones coinciden
            s = torch.from_numpy(SOFT[i].astype(np.float32))
            soft[b, :len(s)] = s.to(logits.device)
        m = inputs["labels"] != -100                   # solo tokens reales (no especiales ni relleno)
        pt = soft[m].clamp_min(1e-8) ** (1 / KD_T)     # profesor con temperatura
        pt = pt / pt.sum(-1, keepdim=True)
        logq = torch.log_softmax(logits[m] / KD_T, -1)  # alumno con temperatura
        kd = (pt * (pt.log() - logq)).sum(-1).mean() * KD_T ** 2
        loss = KD_ALPHA * out.loss + (1 - KD_ALPHA) * kd
        return (loss, out) if return_outputs else loss


def train_student(teacher_runs, tag, seed):
    name = f"{STUDENT_KEY}-kd{tag}-lr{STUDENT_LR:g}-s{seed}"
    if run_done(name):
        print("ya hecho:", name)
        return name
    final, info = train_model(STUDENT_KEY, STUDENT_LR, seed, name, trainer_cls=KDTrainer, ds_hook=kd_dataset(teacher_runs))
    pred = TorchPredictor(final)
    evaluate_all(pred, name, {**info, "fmt": "torch-bf16", "role": "student", "teachers": teacher_runs,
                              "kd_T": KD_T, "kd_alpha": KD_ALPHA})
    del pred; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(Path(final).parent, ignore_errors=True)
    return name


def log_ensemble(name, teacher_runs, sets=("val", "test", "test_long", "test_v2", "test_v3")):
    # Guarda en results.jsonl el resultado del CONJUNTO de profesores (para la tabla de la memoria). Una sola vez.
    if any(r["run"] == name for r in load_results()):
        print("ya registrado:", name)
        return
    for s in sets:
        res = ensemble_eval(rows_for(s), [load_teacher(n, s) for n in teacher_runs])
        rec = {"run": name, "set": s, "fmt": "ensemble", "role": "teacher-ensemble", "members": teacher_runs, **res}
        with open(f"{RUNS_DIR}/results.jsonl", "a") as f:
            f.write(json.dumps(rec) + "\n")
        print(f"{name} {s}: F1 {res['strict_F1']:.3f}  fuga {res['leak_rate']:.3f}")

### 6.2 Profesor A: XLM-R large (2 learning rates con la semilla 42 → el mejor en validación → semillas 13 y 2026)

In [ ]:
# Si la búsqueda de learning rate ya está en results.jsonl, no se repite: solo se reentrenan las 3 semillas con el
# learning rate elegido para volver a tener sus probabilidades (las de la primera ejecución estaban en el disco local).
grid_done = all(val_score(f"{TEACHER_A}-lr{lr:g}-s{SEEDS[0]}") is not None for lr in CANDIDATES[TEACHER_A]["lr_grid"])
if not grid_done:
    for lr in CANDIDATES[TEACHER_A]["lr_grid"]:
        train_teacher(TEACHER_A, lr, SEEDS[0])
lr_a = best_lr_for(TEACHER_A)
print("XLM-R large: learning rate elegido", lr_a)
for seed in SEEDS:
    train_teacher(TEACHER_A, lr_a, seed)
TEACHERS_A = [f"{TEACHER_A}-lr{lr_a:g}-s{s}" for s in SEEDS]
log_ensemble("ensemble-xlmr-large-x3", TEACHERS_A)

### 6.3 Alumno 1: mmBERT-small destilado del conjunto de 3 XLM-R large (3 semillas)

In [ ]:
for seed in SEEDS:
    train_student(TEACHERS_A, "A2", seed)   # "A2": versión corregida (los "A" del 5-nov tenían el fallo de los espacios)
display(show([r for r in load_results() if r["set"] in ("val", "test_v2") and
              (r["run"].startswith(f"{STUDENT_KEY}-kd") or r["run"].startswith(f"{STUDENT_KEY}-lr{STUDENT_LR:g}"))]))

### 6.4 Profesor B: Qwen2.5-7B como etiquetador bidireccional (LoRA, 2 épocas, semilla 42)

In [ ]:
TEACHER_B_RUN = train_teacher(TEACHER_B, CANDIDATES[TEACHER_B]["lr_grid"][0], SEEDS[0], predictor_cls=LLMPredictor)

### 6.5 ¿Qué conjunto de profesores es mejor? Se decide en **validación** (nunca con los tests)

In [ ]:
sets_to_try = {"A": TEACHERS_A, "B": [TEACHER_B_RUN], "A+B": TEACHERS_A + [TEACHER_B_RUN]}
val_f1 = {}
for tag, runs in sets_to_try.items():
    val_f1[tag] = ensemble_eval(val_rows, [load_teacher(n, "val") for n in runs])["strict_F1"]
    print(f"profesores {tag}: strict F1 en validación = {val_f1[tag]:.4f}")
best_tag = max(val_f1, key=val_f1.get)
print("mejor conjunto:", best_tag)
log_ensemble("ensemble-qwen7b", [TEACHER_B_RUN])
log_ensemble("ensemble-xlmr-large-x3+qwen7b", TEACHERS_A + [TEACHER_B_RUN])

### 6.6 Alumno 2 (solo si el mejor conjunto no es A): mmBERT-small destilado del mejor conjunto

In [ ]:
if best_tag != "A":
    for seed in SEEDS:
        train_student(sets_to_try[best_tag], best_tag.replace("+", "") + "2", seed)
else:
    print("El conjunto A sigue siendo el mejor en validación: no hace falta un segundo alumno.")

## 7. Resumen (test_v2): alumnos destilados frente a mmBERT-small normal y a los profesores

In [ ]:
res = load_results()
rows = [r for r in res if r["set"] == "test_v2" and (r.get("role") in ("teacher", "student", "teacher-ensemble")
        or r["run"].startswith(f"{STUDENT_KEY}-lr{STUDENT_LR:g}"))]
display(show(rows))
from google.colab import drive as _d
_d.flush_and_unmount()                               # que todo llegue a Drive antes de cerrar